# Lastni podatkovni tipi

## Okrajšave tipov

In [ ]:
type r3 = float * float * float

type r3 = float * float * float


In [ ]:
type ('k, 'v) slovar = ('k * 'v) list

In [17]:
let vsota_r3 ((x1, y1, z1) : r3) ((x2, y2, z2) : r3) : r3 =
    (x1 +. x2, y1 +. y2)

error: compile_error

## Zapisni tipi

In [21]:
type kompleksno = { re : float; im : float } 

type kompleksno = { re : float; im : float; }


In [22]:
let i  = { re = 0.0; im = 1.0 }

val i : kompleksno = {re = 0.; im = 1.}


In [ ]:
type kompleksno' = { r : float; phi : float }

In [ ]:
let abs z = z.re *. z.re +. z.im *. z.im |> sqrt

In [ ]:
let abs { re; im } = sqrt (re *. re +. im *. im)

val abs : kompleksno -> float = <fun>


## Pametni konstruktorji

In [23]:
type datum = { leto : int; mesec : int; dan : int }

type datum = { leto : int; mesec : int; dan : int; }


In [26]:
{ leto = 2026; mesec = 10; dan = 32 }

- : datum = {leto = 2026; mesec = 10; dan = 32}


In [27]:
let je_prestopno leto =
    (leto mod 4 = 0 && leto mod 100 <> 0) || leto mod 400 = 0

val je_prestopno : int -> bool = <fun>


In [28]:
let dolzina_meseca leto =
    function
    | 4 | 6 | 9 | 11 -> 30
    | 2 -> if je_prestopno leto then 29 else 28
    | _ -> 31

val dolzina_meseca : int -> int -> int = <fun>


In [29]:
let je_veljaven {leto; mesec; dan} =
    let veljaven_dan = 1 <= dan && dan <= dolzina_meseca leto mesec
    and veljaven_mesec = 1 <= mesec && mesec <= 12
    in
    veljaven_dan && veljaven_mesec

val je_veljaven : datum -> bool = <fun>


In [30]:
let naredi_datum leto mesec dan =
    let datum = { leto; mesec; dan } in
    if je_veljaven datum then Some datum else None

val naredi_datum : int -> int -> int -> datum option = <fun>


In [32]:
naredi_datum 2000 10 31

- : datum option = Some {leto = 2000; mesec = 10; dan = 31}


In [ ]:
naredi_datum 1900 2 29

## Zapisni tipi z enim poljem

[Gold Hill, Colorado](https://mathwithbaddrawings.com/2015/08/19/the-smartest-dumb-error-in-the-great-state-of-colorado/)

In [33]:
type temperatura = { kelvin : float }

type temperatura = { kelvin : float; }


In [34]:
let temperatura_iz_kelvinov kelvin =
    if kelvin >= 0. then Some { kelvin } else None

val temperatura_iz_kelvinov : float -> temperatura option = <fun>


In [35]:
let temperatura_iz_celzijev celzij =
    temperatura_iz_kelvinov (celzij +. 273.15)

val temperatura_iz_celzijev : float -> temperatura option = <fun>


In [36]:
let temperatura_iz_fahrenheitov fahrenheit =
    temperatura_iz_celzijev ((fahrenheit -. 32.) *. 5. /. 9.)

val temperatura_iz_fahrenheitov : float -> temperatura option = <fun>


In [39]:
temperatura_iz_celzijev (-300.)

- : temperatura option = None


In [ ]:
temperatura_iz_fahrenheitov 212.

In [ ]:
temperatura_iz_celzijev (-300.)

In [40]:
let voda_vre { kelvin } = kelvin >= 373.15

val voda_vre : temperatura -> bool = <fun>


In [41]:
voda_vre 100.

error: compile_error

## Naštevni tipi

In [42]:
type geometrijski_objekt =
    | Tocka
    | Krog of float
    | Pravokotnik of float * float

type geometrijski_objekt =
    Tocka
  | Krog of float
  | Pravokotnik of float * float


In [45]:
List.map ploscina [Tocka; Pravokotnik (1., 2.); Tocka; Krog 3.]

- : float list = [0.; 2.; 0.; 28.274333882308138]


In [44]:
let ploscina =
    function
    | Tocka -> 0.
    | Krog r -> Float.pi *. r ** 2.
    | Pravokotnik (v, s) -> v *. s

val ploscina : geometrijski_objekt -> float = <fun>


In [ ]:
[Tocka; Pravokotnik (1., 2.); Tocka; Krog 3.]
  |> List.map ploscina
  |> List.fold_left (+.) 0.

In [ ]:
type geometrijski_objekt =
    | Tocka
    | Krog of float
    | Pravokotnik of float * float
    | Trikotnik of float * float * float

In [ ]:
let ploscina =
    function
    | Tocka -> 0.
    | Krog r -> Float.pi *. r ** 2.
    | Pravokotnik (v, s) -> v *. s

In [ ]:
let ploscina =
    function
    | Tocka -> 0.
    | Krog r -> Float.pi *. r ** 2.
    | Pravokotnik (v, s) -> v *. s
    | Trikotnik (a, b, c) ->
        let s = (a +. b +. c) /. 2. in
        sqrt (s *. (s -. a) *. (s -. b) *. (s -. c))

In [ ]:
ploscina (Trikotnik (3., 4., 5.))

## Programi se začnejo s tipi

## Že znani naštevni tipi

In [ ]:
type 'a option =
  | None
  | Some of 'a

In [ ]:
type ('a, 'e) result = Ok of 'a | Error of 'e

In [46]:
let preberi_stevilo niz =
    match int_of_string_opt niz with
    | Some n -> Ok n
    | None -> Error (niz ^ " ni število")

val preberi_stevilo : string -> (int, string) result = <fun>


In [47]:
let preberi_datum niz =
    match String.split_on_char '-' niz with
    | [l; m; d] -> (
        match preberi_stevilo l with
        | Error napaka -> Error napaka
        | Ok leto -> (
            match preberi_stevilo m with
            | Error napaka -> Error napaka
            | Ok mesec -> (
                match preberi_stevilo d with
                | Error napaka -> Error napaka
                | Ok dan -> (
                    match naredi_datum leto mesec dan with
                    | Some datum -> Ok datum
                    | None -> Error (niz ^ " ni veljaven datum")))))
    | _ -> Error (niz ^ " ni oblike leto-mesec-dan")

val preberi_datum : string -> (datum, string) result = <fun>


In [54]:
let ( |> ) x f = f x

let ( >>= ) rezultat f =
    match rezultat with
    | Ok x -> f x
    | Error napaka -> Error napaka

val ( |> ) : 'a -> ('a -> 'b) -> 'b = <fun>


val ( >>= ) : ('a, 'b) result -> ('a -> ('c, 'b) result) -> ('c, 'b) result =
  <fun>


In [ ]:
let preberi_datum niz =
    match String.split_on_char '-' niz with
    | [l; m; d] ->
        preveri_da_je_stevilo l;
        preveri_da_je_stevilo m;
        preveri_da_je_stevilo d;
        let leto = int_of_string l in
        let mesec = int_of_string m in
        let dan = int_of_string d in
        (match naredi_datum leto mesec dan with
         | Some datum -> Ok datum
         | None -> Error (niz ^ " ni veljaven datum"))
    | _ -> Error (niz ^ " ni oblike leto-mesec-dan")

val preberi_datum : string -> (datum, string) result = <fun>


In [48]:
preberi_datum "1991-06-25"

- : (datum, string) result = Ok {leto = 1991; mesec = 6; dan = 25}


In [49]:
preberi_datum "25. 6. 1991"

- : (datum, string) result = Error "25. 6. 1991 ni oblike leto-mesec-dan"


In [50]:
preberi_datum "1991-junij-25"

- : (datum, string) result = Error "junij ni število"


In [52]:
preberi_datum "1980-02-29"

- : (datum, string) result = Ok {leto = 1980; mesec = 2; dan = 29}


## Rekurzivni tipi

In [ ]:
type 'a seznam =
    | Prazen
    | Sestavljen of 'a * 'a seznam

In [56]:
type naravno = Z | S of naravno

type naravno = Z | S of naravno


In [58]:
(* SI predlaga:
   let sestej naravno1 naravno2 =
    match (naravno1, naravno2) with
    | (Z, n) -> n
    | (n, Z) -> n
    | (S n1, S n2) -> S (sestej n1 n2) *)

let rec sestej m n =
  match m with
  | Z -> n
  | S m' -> S (sestej m' n)

let rec zmnozi m n =
  match m with
  | Z -> Z
  | S m' -> sestej (zmnozi m' n) n

val sestej : naravno -> naravno -> naravno = <fun>


val zmnozi : naravno -> naravno -> naravno = <fun>


In [59]:
let ena = S Z

val ena : naravno = S Z


In [60]:
let dva = sestej ena ena

val dva : naravno = S (S Z)


In [61]:
let stiri = zmnozi dva dva

val stiri : naravno = S (S (S (S Z)))


In [63]:
let rec int_of_naravno =
  function
  | Z -> 0
  | S m -> 1 + int_of_naravno m

val int_of_naravno : naravno -> int = <fun>


In [64]:
zmnozi (sestej dva stiri) (sestej ena (sestej dva stiri)) |> int_of_naravno

- : int = 42


<div class="diagram" ondblclick="event.stopPropagation()">
<div class="diagram-step"></div>
<div class="diagram-step" hidden></div>
<div class="diagram-nav"><button type="button" data-d="-1" aria-label="nazaj" onclick="var f = this.closest('.diagram'), s = f.querySelectorAll('.diagram-step'), i = f.querySelector('.diagram-index'), k = i.textContent - 1; s[k].hidden = true; k = (k + s.length + Number(this.dataset.d)) % s.length; s[k].hidden = false; i.textContent = k + 1">&lsaquo;</button> <span class="diagram-index">1</span>/2 <button type="button" data-d="1" aria-label="naprej" onclick="var f = this.closest('.diagram'), s = f.querySelectorAll('.diagram-step'), i = f.querySelector('.diagram-index'), k = i.textContent - 1; s[k].hidden = true; k = (k + s.length + Number(this.dataset.d)) % s.length; s[k].hidden = false; i.textContent = k + 1">&rsaquo;</button></div>
</div>

In [65]:
type 'a drevo =
    | Prazno
    | Sestavljeno of 'a drevo * 'a * 'a drevo

type 'a drevo = Prazno | Sestavljeno of 'a drevo * 'a * 'a drevo


In [66]:
let drevo =
    Sestavljeno (
      Sestavljeno (Prazno, 1, Prazno),
      2,
      Sestavljeno (Sestavljeno (Prazno, 3, Prazno), 4, Prazno)
    )

val drevo : int drevo =
  Sestavljeno (Sestavljeno (Prazno, 1, Prazno), 2,
   Sestavljeno (Sestavljeno (Prazno, 3, Prazno), 4, Prazno))


In [68]:
let rec globina = function
    | Prazno -> 0
    | Sestavljeno (levo, _, desno) -> 1 + max (globina levo) (globina desno)

let rec stevilo_elementov =
    function
    | Prazno -> 0
    | Sestavljeno (levo, koren, desno) -> stevilo_elementov levo + 1 + stevilo_elementov desno

val globina : 'a drevo -> int = <fun>


val stevilo_elementov : 'a drevo -> int = <fun>


In [69]:
globina drevo

- : int = 3


In [70]:
stevilo_elementov drevo

- : int = 4


In [73]:
type vezje =
    | Upor of float
    | Zaporedno of vezje * vezje
    | Vzporedno of vezje * vezje
    | Zarnica

type vezje =
    Upor of float
  | Zaporedno of vezje * vezje
  | Vzporedno of vezje * vezje
  | Zarnica


In [75]:
let rec upornost = function
    | Upor r -> r
    | Zaporedno (vezje1, vezje2) -> upornost vezje1 +. upornost vezje2
    | Vzporedno (vezje1, vezje2) ->
        1. /. (1. /. upornost vezje1 +. 1. /. upornost vezje2)
    | Zarnica -> 0.

val upornost : vezje -> float = <fun>


In [76]:
let preprosto =
    Zaporedno (Upor 1., Zaporedno (Vzporedno (Upor 2., Upor 2.), Upor 3.))

val preprosto : vezje =
  Zaporedno (Upor 1., Zaporedno (Vzporedno (Upor 2., Upor 2.), Upor 3.))


In [77]:
upornost preprosto

- : float = 5.


In [81]:
let zapleteno =
    Vzporedno (Zaporedno (Upor 1., Vzporedno (Zarnica, Upor 6.)), Upor 6.)

val zapleteno : vezje =
  Vzporedno (Zaporedno (Upor 1., Vzporedno (Zarnica, Upor 6.)), Upor 6.)


In [6]:
upornost zapleteno

- : float = 2.


`risanje.ml`

In [87]:
let stevilo_znakov niz =
  String.fold_left
    (fun n bajt -> if Char.code bajt land 0xC0 = 0x80 then n else n + 1)
    0 niz

let vrstice risba = String.split_on_char '\n' risba
let visina risba = List.length (vrstice risba)
let sirina risba = stevilo_znakov (List.hd (vrstice risba))
let ponovi n niz = String.concat "" (List.init n (fun _ -> niz))

let poravnaj v risba =
  let prazne =
    List.init ((v - visina risba) / 2) (fun _ -> ponovi (sirina risba) " ")
  in
  String.concat "\n" (prazne @ vrstice risba @ prazne)

let zaporedno risba1 risba2 =
  let v = max (visina risba1) (visina risba2) in
  List.map2 ( ^ ) (vrstice (poravnaj v risba1)) (vrstice (poravnaj v risba2))
  |> String.concat "\n"

let vzporedno risba1 risba2 =
  let v = max (visina risba1) (visina risba2) in
  let s = max (sirina risba1) (sirina risba2) in
  let veja risba (nad, zica, pod) =
    vrstice (poravnaj v risba)
    |> List.mapi (fun i vrstica ->
        let (levo, desno), polnilo =
          if i < v / 2 then (nad, " ")
          else if i = v / 2 then (zica, "─")
          else (pod, " ")
        in
        let manjka = s - sirina risba in
        let pol_manjka = manjka / 2 in
        levo ^ ponovi (manjka - pol_manjka) polnilo ^ vrstica ^ ponovi pol_manjka polnilo ^ desno)
  in
  String.concat "\n"
    (veja risba1 (("  ", "  "), (" ┌", "┐ "), (" │", "│ "))
    @ [ "─┤" ^ ponovi s " " ^ "├─" ]
    @ veja risba2 ((" │", "│ "), (" └", "┘ "), ("  ", "  ")))

let izpisi risba =
  let brez_presledkov_na_koncu vrstica =
    let rec konec i =
      if i > 0 && vrstica.[i - 1] = ' ' then konec (i - 1) else i
    in
    String.sub vrstica 0 (konec (String.length vrstica))
  in
  risba |> vrstice
  |> List.map brez_presledkov_na_koncu
  |> String.concat "\n" |> print_endline

val stevilo_znakov : string -> int = <fun>


val vrstice : string -> string list = <fun>


val visina : string -> int = <fun>


val sirina : string -> int = <fun>


val ponovi : int -> string -> string = <fun>


val poravnaj : int -> string -> string = <fun>


val zaporedno : string -> string -> string = <fun>


val vzporedno : string -> string -> string = <fun>


val izpisi : string -> unit = <fun>


In [88]:
let rec risba = function
    | Upor r -> Printf.sprintf "─[%g Ω]─" r
    | Zaporedno (vezje1, vezje2) -> zaporedno (risba vezje1) (risba vezje2)
    | Vzporedno (vezje1, vezje2) -> vzporedno (risba vezje1) (risba vezje2)
    | Zarnica -> "─⊗─"

val risba : vezje -> string = <fun>


In [85]:
izpisi (risba preprosto)

        ┌─[2 Ω]─┐
─[1 Ω]──┤       ├──[3 Ω]─
        └─[2 Ω]─┘


- : unit = ()


In [89]:
izpisi (risba zapleteno)

          ┌───⊗───┐
 ┌─[1 Ω]──┤       ├─┐
 │        └─[6 Ω]─┘ │
─┤                  ├─
 │                  │
 └───────[6 Ω]──────┘



- : unit = ()
